In [1]:
import time
import random
import re
import requests
import pandas as pd

VS = "usd"
DAYS = 90
SAMPLE_SIZE = 400
SLEEP_SEC = 15

BASE_LIST  = "https://api.coingecko.com/api/v3/coins/list"
BASE_CHART = "https://api.coingecko.com/api/v3/coins/{id}/market_chart"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; stable-demo/1.0)"
})


def safe_filename(text):
    return re.sub(r'[\\/:*?"<>|]+', "_", text)


def fetch_all_coins():
    r = session.get(BASE_LIST, timeout=30)
    r.raise_for_status()
    return r.json()


def fetch_market_chart(coin_id, days=90, vs="usd", max_retry=3):

    url = BASE_CHART.format(id=coin_id)
    params = {"vs_currency": vs, "days": days}

    for attempt in range(max_retry):

        r = session.get(url, params=params, timeout=30)

        if r.status_code == 200:

            data = r.json()

            if "prices" not in data:
                raise RuntimeError("prices missing")

            df = pd.DataFrame(data["prices"], columns=["timestamp","price"])

            df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

            return df


        if r.status_code == 429:

            wait = 5 * (attempt + 1)
            print(f"[429] wait {wait}s")
            time.sleep(wait)
            continue

        r.raise_for_status()

    raise RuntimeError("retry failed")


def sec_to_hms(sec):

    m = int(sec // 60)
    s = int(sec % 60)

    return f"{m}m{s}s"


def main():

    start_time = time.time()

    coins = fetch_all_coins()

    print("total coins:", len(coins))

    coins_sample = random.sample(coins, min(SAMPLE_SIZE, len(coins)))

    success = 0
    fail = 0

    total = len(coins_sample)

    for i, coin in enumerate(coins_sample, start=1):

        coin_id = coin["id"]
        symbol  = coin["symbol"]
        name    = coin["name"]

        print(f"\n[{i}/{total}] Fetching {name} ({symbol})")

        try:

            df = fetch_market_chart(coin_id, DAYS, VS)

            safe_symbol = safe_filename(symbol.upper())
            safe_id     = safe_filename(coin_id)

            filename = f"{i:03d}_{safe_symbol}_{safe_id}_last{DAYS}d_{VS}.csv"

            df.to_csv(filename, index=False)

            print("  -> saved:", filename)

            success += 1

        except Exception as e:

            print("  -> error:", e)

            fail += 1


        elapsed = time.time() - start_time

        print(
            f"progress: {i}/{total}  success={success}  fail={fail}  elapsed={sec_to_hms(elapsed)}"
        )

        time.sleep(SLEEP_SEC)


    print("\n===== finished =====")
    print("success:", success)
    print("fail:", fail)


if __name__ == "__main__":
    main()

total coins: 18361

[1/400] Fetching DeLorean (dmc)
  -> saved: 001_DMC_delorean_last90d_usd.csv
progress: 1/400  success=1  fail=0  elapsed=0m5s

[2/400] Fetching Depinsim (esim)
  -> saved: 002_ESIM_depinsim_last90d_usd.csv
progress: 2/400  success=2  fail=0  elapsed=0m21s

[3/400] Fetching USDCx (Stacks) (usdcx)
  -> saved: 003_USDCX_usdcx-stacks_last90d_usd.csv
progress: 3/400  success=3  fail=0  elapsed=0m36s

[4/400] Fetching Froggy (frog)
  -> saved: 004_FROG_froggy-2_last90d_usd.csv
progress: 4/400  success=4  fail=0  elapsed=0m53s

[5/400] Fetching Echo Protocol (echo)
  -> saved: 005_ECHO_echo-protocol_last90d_usd.csv
progress: 5/400  success=5  fail=0  elapsed=1m9s

[6/400] Fetching USDT0 (usdt0)
  -> saved: 006_USDT0_usdt0_last90d_usd.csv
progress: 6/400  success=6  fail=0  elapsed=1m26s

[7/400] Fetching CoNET Network (conet)
  -> saved: 007_CONET_conet-network_last90d_usd.csv
progress: 7/400  success=7  fail=0  elapsed=1m41s

[8/400] Fetching Hive Game Token (hgt)
  -> sa